# ARS Session 1 — RFT on the Current Architecture

This notebook runs the D3 Session 1 sequence: generation validation, Gate 0, positive RFT rollout collection, Stage 1 RFT training, and paired pure-steering evaluation. The route is `[17, 20, 23, 26]`, the magnitude head is disabled (`m ≡ 1`), the Phi-2 backbone stays frozen, and training uses clean completion-token mean NLL. Stage 2 / redesigned SteerNet experiments are intentionally not part of this session.

Run cells from top to bottom on a Kaggle **Dual T4** accelerator. The full benchmark uses all 1,319 GSM8K test questions at 0-shot and 2-shot with greedy decoding and repetition penalty 1.0.

## 0. Configure Kaggle runtime and checkout

Set allocator and tokenizer environment variables before launching Python worker processes. The checkout is refreshed from the `development` branch so notebook commands match the committed code.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Repository: {REPO_DIR} ({BRANCH})")

## 1. Verify the Kaggle accelerator and repository tests

The two collection workers each get one T4. Their per-process CUDA device is remapped to `cuda:0` by `CUDA_VISIBLE_DEVICES`. Training/evaluation use one T4 and the configured micro-batch size of 2.

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
gpu_rows = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
print("Visible GPUs:", gpu_rows)
if len(gpu_rows) < 2 or any("T4" not in row for row in gpu_rows[:2]):
    raise RuntimeError("Select Kaggle's Dual T4 accelerator before continuing")

subprocess.run([sys.executable, "-m", "pytest", "tests/", "-x", "-q"], check=True)

## 2. Assert the Stage 1 configuration

These checks make the session locks explicit before any model is loaded. `max_tok_len=768`, collection batch size 2, gradient accumulation 16, and no generation batching above 2 keep memory use bounded on T4.

In [ ]:
from src.configs.config import RSCConfig

cfg = RSCConfig()
assert cfg.fixed_layers == [17, 20, 23, 26]
assert cfg.steernet_use_mag_head is False  # m ≡ 1
assert cfg.use_gate is False
assert cfg.phase1_lr == 1.5e-4
assert cfg.objective == "nll"
assert cfg.max_tok_len == 768
assert cfg.batch_size == 2 and cfg.grad_accum == 16
assert cfg.generation_repetition_penalty == 1.0
cfg.validate()
print({
    "route": cfg.fixed_layers,
    "mag_head": cfg.steernet_use_mag_head,
    "loss": "positive completion token-mean NLL",
    "phase1_lr": cfg.phase1_lr,
    "max_tok_len": cfg.max_tok_len,
    "micro_batch": cfg.batch_size,
    "grad_accum": cfg.grad_accum,
    "use_gate": cfg.use_gate,
})

## 3. Step 1 — Batched vs. batch-1 generation equivalence

This creates `report/rft/generation_equivalence.json`. The pipeline blocks later collection/training unless the configured ≤1.5 percentage-point accuracy gap, ≥90% string identity, and aggregate pass status succeed.

In [ ]:
RFT_DIR = Path("report/rft")
RFT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, "scripts/run_rft_pipeline.py",
    "--stage", "equivalence",
    "--output_dir", str(RFT_DIR),
], check=True)

## 4. Step 2 — Gate 0 sampling sanity probe

Gate 0 compares 8 sampled continuations at temperature 0.7 with paired greedy decoding on up to 60 questions. It stops the run unless sampled lenient pass@8 is at least 70% and beats greedy accuracy.

In [ ]:
subprocess.run([
    sys.executable, "scripts/run_rft_pipeline.py",
    "--stage", "gate0",
    "--k", "8",
    "--temperature", "0.7",
    "--output_dir", str(RFT_DIR),
], check=True)
import json

gate0 = json.loads((RFT_DIR / "gate0_stats.json").read_text(encoding="utf-8"))
assert gate0["passed"] and gate0["pass_at_k_lenient"] >= 0.70
print({key: gate0[key] for key in ("n", "pass_at_k_lenient", "greedy_lenient", "pass_at_k_delta_vs_greedy_pp", "passed")})

## 5. Step 3 — Collect positive RFT rollouts on both T4s

Each worker receives a disjoint shard of the same seeded 2,000-question sample and writes its own JSONL/stat files. Workers are launched as separate processes so each sees only one T4 and initializes CUDA after device selection. Both must finish successfully; each shard enforces the ≥15% verified-positive yield gate.

In [ ]:
import time

collection_env = os.environ.copy()
collection_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
collection_env["TOKENIZERS_PARALLELISM"] = "false"
collection_env["PYTHONUNBUFFERED"] = "1"
workers = []
for shard, gpu in ((0, "0"), (1, "1")):
    env = collection_env.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu
    log_path = RFT_DIR / f"collect_shard{shard}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    command = [
        sys.executable, "scripts/run_rft_pipeline.py",
        "--stage", "collect",
        "--n_questions", "2000",
        "--k", "8",
        "--temperature", "0.7",
        "--n_shards", "2",
        "--shard", str(shard),
        "--output_dir", str(RFT_DIR),
    ]
    process = subprocess.Popen(command, env=env, stdout=log_handle, stderr=subprocess.STDOUT)
    workers.append((shard, process, log_handle, log_path))

failures = []
for shard, process, log_handle, log_path in workers:
    return_code = process.wait()
    log_handle.close()
    if return_code != 0:
        tail = log_path.read_text(encoding="utf-8", errors="replace")[-6000:]
        failures.append(f"shard {shard} exited {return_code}:\n{tail}")
if failures:
    raise RuntimeError("\n\n".join(failures))

shard_stats = [json.loads((RFT_DIR / f"shard{i}_stats.json").read_text(encoding="utf-8")) for i in (0, 1)]
print({"shard0": shard_stats[0], "shard1": shard_stats[1]})

## 6. Step 4 — Train Stage 1 RFT

Training combines the two rollout shards and samples mixed 0/2/4-shot contexts. It uses the locked fixed route, identity magnitude, frozen Phi-2, and positive-completion token-mean NLL. The trainer uses micro-batch 2 with gradient accumulation 16, gradient checkpointing, and a maximum 768-token sequence length. The resulting adapter is `checkpoints/steernet.pt`.

In [ ]:
train_env = os.environ.copy()
train_env["CUDA_VISIBLE_DEVICES"] = "0"
train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
train_env["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run([
    sys.executable, "scripts/run_rft_pipeline.py",
    "--stage", "train",
    "--epochs", "2",
    "--checkpoint", "checkpoints/steernet.pt",
    "--output_dir", str(RFT_DIR),
], env=train_env, check=True)
assert Path("checkpoints/steernet.pt").is_file()

## 7. Step 5 — Calibrate and evaluate pure steering vs. baseline

The deployment contract is calibrated from validation data using the trained checkpoint. The paired benchmark then evaluates baseline and the isolated `fixed` steering arm on all 1,319 GSM8K test questions at both 0-shot and 2-shot. Evaluation is greedy, uses repetition penalty 1.0, one question per generation, and reloads the dedicated fixed-arm checkpoint. No gate/router arm is enabled.

In [ ]:
eval_env = os.environ.copy()
eval_env["CUDA_VISIBLE_DEVICES"] = "0"
eval_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
eval_env["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run([
    sys.executable, "scripts/run_ars_pipeline.py",
    "--stage", "S5",
    "--s_fixed", "17,20,23,26",
    "--calib_sel_n", "128",
    "--contract", "deployment_contract.json",
], env=eval_env, check=True)
subprocess.run([
    sys.executable, "scripts/run_ars_pipeline.py",
    "--stage", "bench",
    "--s_fixed", "17,20,23,26",
    "--contract", "deployment_contract.json",
    "--n_test", "1319",
    "--shots", "0", "2",
    "--arms", "baseline,fixed",
    "--record", "report/phase1_gsm8k_1319.json",
    "--report",
], env=eval_env, check=True)

## 8. Review paired results and save artifacts

The JSON record contains per-question correctness and aggregate strict/lenient accuracy. This cell prints the full-test-set lenient scores and the paired fixed-minus-baseline delta for each shot condition.

In [ ]:
record_path = Path("report/phase1_gsm8k_1319.json")
record = json.loads(record_path.read_text(encoding="utf-8"))
summary = {}
for shots, result in record["shots"].items():
    baseline = result["baseline"]["summary"]
    fixed = result["fixed"]["summary"]
    summary[shots] = {
        "n": baseline["n"],
        "baseline_lenient_accuracy": baseline["lenient_accuracy"],
        "fixed_lenient_accuracy": fixed["lenient_accuracy"],
        "paired_delta_pp": fixed["lenient_accuracy"] - baseline["lenient_accuracy"],
    }
    assert baseline["n"] == fixed["n"] == 1319
print(json.dumps(summary, indent=2))
print("Artifacts:", record_path, Path("checkpoints/steernet.pt"), Path("deployment_contract.json"))